# Live pulsoximeter-måling

Denne notebook er en simpel prototype til live-demonstration. Den gentager samme målecyklus igen og igen:

| Måling | 660 nm LED | 940 nm LED |
|---|---|---|
| Måling 1 | Tændt | Slukket |
| Måling 2 | Slukket | Tændt |
| Måling 3 | Slukket | Slukket |

Mørkemålingen er bare analogmåling med begge LED'er slukket. Den bruges som offset/baggrund og trækkes fra LED-målingerne.

## Tilslutning

| Signal | Discovery 3 | Kredsløb |
|---|---|---|
| 660 nm gate | DIO0 | MOSFET-gate for rød LED |
| 940 nm gate | DIO1 | MOSFET-gate for IR LED |
| Analog måling + | Scope CH1 + | Udgang fra sidste analogtrin |
| Analog måling - | Scope CH1 - / GND-reference | Reference/GND ved sidste analogtrin |

Notebooken gemmer ikke målinger automatisk. Den er tænkt som live-visning, så OBS/screen capture er nok til demo.

Hvis + og - proberne sidder på udgangen af sidste trin og dets reference, bør live-plottet vise PPG-variationer, hvis fingeren er stabilt placeret, LED-strømmen er passende, og analogsignalet ikke klipper.

In [1]:
from dataclasses import dataclass
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

import dwfpy as dwf
from dwfpy.constants import TriggerSource

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.35,
})


In [2]:
@dataclass
class MaaleConfig:
    sample_rate_hz: float = 20_000
    fase_varighed_s: float = 0.08
    settle_s: float = 0.01
    total_varighed_s: float = 90
    analog_channel: int = 0
    analog_range_v: float = 2.0
    dio_660: int = 0
    dio_940: int = 1
    spo2_a: float = -24.87
    spo2_b: float = 113.8
    puls_vindue_s: float = 60
    min_puls_tid_s: float = 15
    spo2_vindue_s: float = 30
    spo2_min_dc_v: float = 1e-3
    spo2_min_ac_v: float = 1e-5
    min_bpm: float = 40
    max_bpm: float = 140


cfg = MaaleConfig()
print(f"Fasevarighed: {cfg.fase_varighed_s * 1000:.0f} ms")
print(f"Cykler pr. sekund: {1 / (3 * (cfg.fase_varighed_s + cfg.settle_s)):.1f}")


Fasevarighed: 80 ms
Cykler pr. sekund: 3.7


## Hardwarefunktioner

LED'erne styres kun som tænd/sluk-signaler på DIO-linjerne. For hver fase sættes LED-tilstanden, der ventes kort på at kredsløbet falder til ro, og Scope CH1 optager et kort spændingsvindue.

In [3]:
def setup_scope(device, cfg):
    scope = device.analog_input
    scope.reset()
    scope.trigger.source = TriggerSource.NONE

    for channel_index, _ in enumerate(scope.channels):
        scope.setup_channel(
            channel_index,
            range=cfg.analog_range_v,
            offset=0.0,
            coupling="dc",
            filter="average",
            enabled=(channel_index == cfg.analog_channel),
        )


def set_leds(device, cfg, led_660=False, led_940=False):
    pattern = device.digital_output
    pattern[cfg.dio_660].setup_constant("high" if led_660 else "low", output_mode="push-pull", idle_state="low")
    pattern[cfg.dio_940].setup_constant("high" if led_940 else "low", output_mode="push-pull", idle_state="low")
    pattern.configure(start=True)


def measure_voltage(device, cfg):
    scope = device.analog_input
    recorder = scope.record(
        sample_rate=cfg.sample_rate_hz,
        length=cfg.fase_varighed_s,
        buffer_size=8192,
        configure=True,
        start=True,
    )
    samples = recorder.channels[cfg.analog_channel].data_samples
    return float(np.mean(samples))


def measure_cycle(device, cfg):
    set_leds(device, cfg, led_660=True, led_940=False)
    time.sleep(cfg.settle_s)
    v_660 = measure_voltage(device, cfg)

    set_leds(device, cfg, led_660=False, led_940=True)
    time.sleep(cfg.settle_s)
    v_940 = measure_voltage(device, cfg)

    set_leds(device, cfg, led_660=False, led_940=False)
    time.sleep(cfg.settle_s)
    v_dark = measure_voltage(device, cfg)

    return {
        "v_660": v_660,
        "v_940": v_940,
        "v_dark": v_dark,
        "v_660_korr": v_660 - v_dark,
        "v_940_korr": v_940 - v_dark,
    }


## LED pulse-only test

Denne sektion måler ikke noget analogt. Den skifter kun LED'erne i samme rækkefølge som målecyklussen, så du kan tjekke at DIO0/Lead 0 driver 660 nm LED'en, og DIO1/Lead 1 driver 940 nm LED'en.

In [4]:
def led_pulse_only_test(cfg, cycles=20, fase_varighed_s=0.25):
    """Kører kun LED-multipleksing: 660 on, 940 on, begge off."""
    print("Starter LED-test")
    print(f"DIO{cfg.dio_660}/Lead 0: 660 nm")
    print(f"DIO{cfg.dio_940}/Lead 1: 940 nm")

    with dwf.Device() as device:
        try:
            for cycle in range(cycles):
                print(f"Cyklus {cycle + 1}/{cycles}: 660 nm ON, 940 nm OFF", end="\r")
                set_leds(device, cfg, led_660=True, led_940=False)
                time.sleep(fase_varighed_s)

                print(f"Cyklus {cycle + 1}/{cycles}: 660 nm OFF, 940 nm ON", end="\r")
                set_leds(device, cfg, led_660=False, led_940=True)
                time.sleep(fase_varighed_s)

                print(f"Cyklus {cycle + 1}/{cycles}: begge LED'er OFF       ", end="\r")
                set_leds(device, cfg, led_660=False, led_940=False)
                time.sleep(fase_varighed_s)
        finally:
            set_leds(device, cfg, led_660=False, led_940=False)
            print("\nLED-test færdig. Begge LED'er er slukket.")


KOR_LED_TEST = False

if KOR_LED_TEST:
    led_pulse_only_test(cfg, cycles=20, fase_varighed_s=0.25)
else:
    print("LED-test klar. Sæt KOR_LED_TEST = True for at teste DIO0/DIO1.")


LED-test klar. Sæt KOR_LED_TEST = True for at teste DIO0/DIO1.


## SpO2 og puls

Pulsestimatet er med vilje simpelt: find toppe eller dale i den seneste 940 nm-kurve og beregn BPM fra tiden mellem dem.

SpO2 kræver både AC- og DC-led for 660 nm og 940 nm. Hvis proben sidder efter et højpas-/båndpasled, kan DC-leddet være fjernet. I så fald er ratio-of-ratios ikke fysisk gyldig, og notebooken viser `--` i stedet for et falsk procenttal.

In [5]:
def cfg_value(cfg, name, default):
    return getattr(cfg, name, default)


def calculate_spo2(data, cfg):
    spo2_vindue_s = cfg_value(cfg, "spo2_vindue_s", 30)
    spo2_min_dc_v = cfg_value(cfg, "spo2_min_dc_v", 1e-3)
    spo2_min_ac_v = cfg_value(cfg, "spo2_min_ac_v", 1e-5)

    recent = data[data["t_s"] >= data["t_s"].max() - spo2_vindue_s]
    if len(recent) < 8:
        return np.nan, np.nan, "venter på mere data"

    ac_660 = np.percentile(recent["v_660_korr"], 95) - np.percentile(recent["v_660_korr"], 5)
    ac_940 = np.percentile(recent["v_940_korr"], 95) - np.percentile(recent["v_940_korr"], 5)
    dc_660 = abs(recent["v_660_korr"].mean())
    dc_940 = abs(recent["v_940_korr"].mean())

    if dc_660 < spo2_min_dc_v or dc_940 < spo2_min_dc_v:
        return np.nan, np.nan, "SpO2 ikke gyldig: DC-led for lavt"
    if ac_660 < spo2_min_ac_v or ac_940 < spo2_min_ac_v:
        return np.nan, np.nan, "SpO2 ikke gyldig: AC-led for lavt"

    R = (ac_660 / dc_660) / (ac_940 / dc_940)
    spo2 = cfg.spo2_a * R + cfg.spo2_b

    if not np.isfinite(R) or not np.isfinite(spo2) or spo2 < 50 or spo2 > 100:
        return R, np.nan, "SpO2 uden for gyldigt område"

    return R, spo2, "ok"


def smooth(y, width=3):
    if len(y) < width:
        return y
    kernel = np.ones(width) / width
    return np.convolve(y, kernel, mode="same")


def turning_points(t, y, cfg, polarity=1):
    if len(y) >= 4:
        trend = np.polyval(np.polyfit(t - t[0], y, 1), t - t[0])
        y = y - trend
    y = polarity * smooth(y - np.median(y), width=3)
    if len(y) < 3 or np.std(y) < 1e-12:
        return np.array([], dtype=int)

    threshold = 0.60 * np.std(y)
    max_bpm = cfg_value(cfg, "max_bpm", 140)
    min_distance_s = 60 / max_bpm
    points = []
    last_i = None

    for i in range(1, len(y) - 1):
        is_peak = y[i] > y[i - 1] and y[i] >= y[i + 1] and y[i] > threshold
        if not is_peak:
            continue

        if last_i is not None and t[i] - t[last_i] < min_distance_s:
            if y[i] > y[last_i]:
                points[-1] = i
                last_i = i
            continue

        points.append(i)
        last_i = i

    return np.array(points, dtype=int)


def bpm_from_points(t, points, cfg):
    if len(points) < 3:
        return np.nan, np.inf

    min_bpm = cfg_value(cfg, "min_bpm", 40)
    max_bpm = cfg_value(cfg, "max_bpm", 140)

    intervals = np.diff(t[points])
    valid = (intervals >= 60 / max_bpm) & (intervals <= 60 / min_bpm)
    intervals = intervals[valid]
    if len(intervals) < 2:
        return np.nan, np.inf

    bpm = 60 / np.median(intervals)
    regularity = np.std(intervals) / np.median(intervals)
    if bpm < min_bpm or bpm > max_bpm:
        return np.nan, np.inf
    return bpm, regularity


def calculate_bpm(data, cfg):
    puls_vindue_s = cfg_value(cfg, "puls_vindue_s", 60)
    min_puls_tid_s = cfg_value(cfg, "min_puls_tid_s", 15)

    recent = data[data["t_s"] >= data["t_s"].max() - puls_vindue_s]
    if len(recent) < 6 or recent["t_s"].max() - recent["t_s"].min() < min_puls_tid_s:
        return np.nan, np.array([], dtype=int), "", "venter på flere pulsslag"

    t = recent["t_s"].to_numpy()
    y = recent["v_940_korr"].to_numpy()

    peak_local = turning_points(t, y, cfg, polarity=1)
    trough_local = turning_points(t, y, cfg, polarity=-1)

    peak_bpm, peak_regularity = bpm_from_points(t, peak_local, cfg)
    trough_bpm, trough_regularity = bpm_from_points(t, trough_local, cfg)

    if np.isfinite(trough_bpm) and (not np.isfinite(peak_bpm) or trough_regularity < peak_regularity):
        chosen = trough_local
        bpm = trough_bpm
        kind = "dale"
        regularity = trough_regularity
    else:
        chosen = peak_local
        bpm = peak_bpm
        kind = "toppe"
        regularity = peak_regularity

    if not np.isfinite(bpm):
        return np.nan, np.array([], dtype=int), "", "ingen stabile toppe/dale"
    if regularity > 0.35:
        return np.nan, recent.index.to_numpy()[chosen], kind, "pulsen er for ujævn til sikkert HR-tal"

    return bpm, recent.index.to_numpy()[chosen], kind, "ok"


## Live-plot

Første graf viser de mørkekorrigerede LED-målinger. Anden graf viser puls og SpO2 live. Toppe/dale markeres på 940 nm-signalet.

In [6]:
def fmt(value, suffix=""):
    return "--" if not np.isfinite(value) else f"{value:.1f}{suffix}"


def show_live(data, metrics, pulse_indices, pulse_kind):
    clear_output(wait=True)
    fig, ax = plt.subplots(2, 1, figsize=(13, 7), dpi=120, constrained_layout=True)

    recent = data[data["t_s"] >= data["t_s"].max() - 15]
    ax[0].plot(recent["t_s"], recent["v_660_korr"], color="tab:red", label="660 nm")
    ax[0].plot(recent["t_s"], recent["v_940_korr"], color="tab:green", label="940 nm")

    visible = [idx for idx in pulse_indices if idx in recent.index]
    if visible:
        ax[0].scatter(data.loc[visible, "t_s"], data.loc[visible, "v_940_korr"], color="black", s=28, label=f"Puls-{pulse_kind}")

    last = metrics[-1]
    ax[0].set_title(f"Live signal | HR {fmt(last['bpm'], ' BPM')} | SpO2 {fmt(last['spo2'], ' %')}")
    notes = []
    if last.get("bpm_note") != "ok":
        notes.append(f"HR: {last.get('bpm_note')}")
    if last.get("spo2_note") != "ok":
        notes.append(f"SpO2: {last.get('spo2_note')}")
    if notes:
        ax[0].text(
            0.01,
            0.98,
            "\n".join(notes),
            transform=ax[0].transAxes,
            va="top",
            ha="left",
            fontsize=10,
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "0.8"},
        )
    ax[0].set_xlabel("Tid [s]")
    ax[0].set_ylabel("Mørkekorrigeret spænding [V]")
    ax[0].grid(True)
    ax[0].legend(frameon=False, loc="upper right")

    metrics_df = pd.DataFrame(metrics)
    ax[1].plot(metrics_df["t_s"], metrics_df["bpm"], color="tab:blue", label="HR [BPM]")
    ax[1].set_ylabel("HR [BPM]")
    ax[1].set_ylim(cfg.min_bpm - 10, cfg.max_bpm + 10)
    ax[1].grid(True)

    ax2 = ax[1].twinx()
    ax2.plot(metrics_df["t_s"], metrics_df["spo2"], color="tab:purple", label="SpO2 [%]")
    ax2.set_ylabel("SpO2 [%]")
    ax2.set_ylim(80, 100)

    handles1, labels1 = ax[1].get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax[1].legend(handles1 + handles2, labels1 + labels2, frameon=False, loc="upper right")
    ax[1].set_xlabel("Tid [s]")

    display(fig)
    plt.close(fig)


def run_live(cfg):
    rows = []
    metrics = []
    start = time.monotonic()

    with dwf.Device() as device:
        setup_scope(device, cfg)
        set_leds(device, cfg, led_660=False, led_940=False)

        try:
            while time.monotonic() - start < cfg.total_varighed_s:
                row = measure_cycle(device, cfg)
                row["t_s"] = time.monotonic() - start
                rows.append(row)

                data = pd.DataFrame(rows)
                R, spo2, spo2_note = calculate_spo2(data, cfg)
                bpm, pulse_indices, pulse_kind, bpm_note = calculate_bpm(data, cfg)

                metrics.append({"t_s": row["t_s"], "R": R, "spo2": spo2, "bpm": bpm, "spo2_note": spo2_note, "bpm_note": bpm_note})
                show_live(data, metrics, pulse_indices, pulse_kind)

        finally:
            set_leds(device, cfg, led_660=False, led_940=False)

    return pd.DataFrame(rows), pd.DataFrame(metrics)


## Start måling

Sæt `KOR_LIVE_MAALING = True` når Discovery 3 og kredsløbet er tilsluttet.

In [7]:
KOR_LIVE_MAALING = False

if KOR_LIVE_MAALING:
    cfg.total_varighed_s = 90
    maalinger, live_metrics = run_live(cfg)
else:
    print("Klar. Sæt KOR_LIVE_MAALING = True for at starte live-visningen.")


Klar. Sæt KOR_LIVE_MAALING = True for at starte live-visningen.
